# Figure 5 RSSI CDF Pipeline

This notebook scans the HRBB AP-sweep artifacts, extracts the final method snapshots, renders one coverage map per method, exports the RSSI samples used for the CDF, and writes a summary JSON with the extracted configuration and statistics.

In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any, Mapping, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.ndimage import gaussian_filter1d
from sionna.rt import RadioMapSolver

from reflector_position.metrics import POWER_EPSILON, rss_to_dbm
from reflector_position.optimizers.memetic.memetic_plotting import (
    _apply_snapshot_to_scene,
    _render_coverage_snapshot,
)
from reflector_position.scene_setup import setup_building_floor_scene
from reflector_position.optimizers.memetic.demand_weights import generate_spatial_priority_map


repo_root = Path("/home/hieule/research/reflector-position")
artifacts_dir = Path(
    "/home/hieule/research/reflector-position/tmp_comparison_results2/hrbb/per_trial_runs/aps_03_seed_0044/artifacts"
)
run_dir = artifacts_dir.parent

output_root = repo_root / "output" / "figure5-hrbb"
coverage_dir = output_root / "coverage_maps"
output_root.mkdir(parents=True, exist_ok=True)
coverage_dir.mkdir(parents=True, exist_ok=True)


def _load_json(path: Path) -> Optional[dict[str, Any]]:
    if not path.exists():
        return None
    return json.loads(path.read_text())


def _json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


def _is_sequence(value: Any) -> bool:
    return isinstance(value, Sequence) and not isinstance(value, (str, bytes, bytearray))


def _as_float(value: Any) -> Optional[float]:
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return None
    if not np.isfinite(numeric):
        return None
    return numeric


def _first_present(container: Mapping[str, Any], *keys: str) -> Any:
    for key in keys:
        if key in container:
            value = container.get(key)
            if value is not None:
                return value
    return None


def _coerce_triplet(raw_value: Any, fallback_z: Optional[float] = None) -> Optional[list[float]]:
    if not _is_sequence(raw_value) or len(raw_value) < 2:
        return None
    try:
        x_value = float(raw_value[0])
        y_value = float(raw_value[1])
        z_value = float(raw_value[2]) if len(raw_value) >= 3 else float(3.8 if fallback_z is None else fallback_z)
    except (TypeError, ValueError):
        return None
    return [x_value, y_value, z_value]


def _normalize_positions(raw_positions: Any, fallback_z: Optional[float]) -> Optional[list[list[float]]]:
    if not _is_sequence(raw_positions) or len(raw_positions) == 0:
        return None
    normalized: list[list[float]] = []
    for raw_position in raw_positions:
        triplet = _coerce_triplet(raw_position, fallback_z=fallback_z)
        if triplet is not None:
            normalized.append(triplet)
    return normalized or None


def _normalize_directions(raw_directions: Any) -> Optional[list[list[float]]]:
    if not _is_sequence(raw_directions) or len(raw_directions) == 0:
        return None
    normalized: list[list[float]] = []
    for raw_direction in raw_directions:
        if not _is_sequence(raw_direction) or len(raw_direction) < 2:
            continue
        try:
            dx_value = float(raw_direction[0])
            dy_value = float(raw_direction[1])
            dz_value = float(raw_direction[2]) if len(raw_direction) >= 3 else 0.0
        except (TypeError, ValueError):
            continue
        normalized.append([dx_value, dy_value, dz_value])
    return normalized or None


def _normalize_reflector(container: Mapping[str, Any]) -> Optional[dict[str, Any]]:
    raw_reflector = container.get("reflector")
    reflector: dict[str, Any] = {}

    if isinstance(raw_reflector, Mapping):
        for key in ("u", "v"):
            if key in raw_reflector:
                value = _as_float(raw_reflector.get(key))
                if value is not None:
                    reflector[key] = value
        if "target" in raw_reflector:
            target = _coerce_triplet(raw_reflector.get("target"), fallback_z=1.5)
            if target is not None:
                reflector["target"] = target
        elif all(key in raw_reflector for key in ("focal_x", "focal_y", "focal_z")):
            target = _coerce_triplet(
                [raw_reflector.get("focal_x"), raw_reflector.get("focal_y"), raw_reflector.get("focal_z")],
                fallback_z=1.5,
            )
            if target is not None:
                reflector["target"] = target
        return reflector or None

    u_value = _as_float(container.get("reflector_u"))
    v_value = _as_float(container.get("reflector_v"))
    if u_value is not None:
        reflector["u"] = u_value
    if v_value is not None:
        reflector["v"] = v_value

    target = _first_present(container, "reflector_target", "initial_focal_point", "focal_point")
    if target is None and all(key in container for key in ("focal_x", "focal_y", "focal_z")):
        target = [container["focal_x"], container["focal_y"], container["focal_z"]]
    target_triplet = _coerce_triplet(target, fallback_z=1.5)
    if target_triplet is not None:
        reflector["target"] = target_triplet

    return reflector or None


def _extract_direct_snapshot(container: Mapping[str, Any], fixed_z: Optional[float]) -> Optional[dict[str, Any]]:
    raw_positions = _first_present(container, "positions", "ap_positions", "final_positions", "best_positions")
    raw_directions = _first_present(container, "directions", "ap_directions", "final_directions", "best_directions")
    raw_reflector = container.get("reflector")

    if raw_positions is None and raw_directions is None and raw_reflector is None:
        return None

    positions = _normalize_positions(raw_positions, fallback_z=fixed_z)
    if positions is None:
        return None

    snapshot: dict[str, Any] = {"positions": positions}
    directions = _normalize_directions(raw_directions)
    if directions is not None:
        snapshot["directions"] = directions
    reflector = _normalize_reflector(container)
    if reflector is not None:
        snapshot["reflector"] = reflector
    return snapshot


def _extract_snapshot(container: Any, fixed_z: Optional[float], depth: int = 0) -> Optional[dict[str, Any]]:
    """Extract method snapshot preferring final iteration/configuration fields.

    Priority order:
    1) explicit final config nodes,
    2) best config nodes,
    3) top-level best_* fields,
    4) fallback recursive scan.
    """
    if not isinstance(container, Mapping) or depth > 8:
        return None

    # Highest priority: explicit final configuration payloads.
    for key in ("final_configuration",):
        nested = container.get(key)
        if isinstance(nested, Mapping):
            snapshot = _extract_direct_snapshot(nested, fixed_z=fixed_z)
            if snapshot is not None:
                return snapshot

    # Common nested result containers that may include final/best config.
    for key in ("results", "global_best_result", "best_configuration"):
        nested = container.get(key)
        if isinstance(nested, Mapping):
            snapshot = _extract_snapshot(nested, fixed_z=fixed_z, depth=depth + 1)
            if snapshot is not None:
                return snapshot

    # Memetic payload can hold final configuration under global_best_result.results.
    global_best = container.get("global_best_result")
    if isinstance(global_best, Mapping):
        nested_results = global_best.get("results")
        if isinstance(nested_results, Mapping):
            snapshot = _extract_snapshot(nested_results, fixed_z=fixed_z, depth=depth + 1)
            if snapshot is not None:
                return snapshot

    # Standard method result payloads expose best_* at top level.
    direct_snapshot = _extract_direct_snapshot(container, fixed_z=fixed_z)
    if direct_snapshot is not None:
        return direct_snapshot

    # Last resort: recurse through sequences/mappings.
    for key in ("seeds", "hall_of_fame", "generation_details", "top_individuals"):
        nested = container.get(key)
        if _is_sequence(nested):
            # Prefer latest entries first to avoid first-iteration/seed bias.
            for item in list(nested)[::-1]:
                snapshot = _extract_snapshot(item, fixed_z=fixed_z, depth=depth + 1)
                if snapshot is not None:
                    return snapshot

    for value in container.values():
        if isinstance(value, Mapping):
            snapshot = _extract_snapshot(value, fixed_z=fixed_z, depth=depth + 1)
            if snapshot is not None:
                return snapshot

    return None


def _discover_method_payloads(directory: Path) -> dict[str, dict[str, Any]]:
    payloads: dict[str, dict[str, Any]] = {}
    for path in sorted(directory.glob("*_results.json")):
        method_name = path.name[: -len("_results.json")]
        payloads[method_name] = json.loads(path.read_text())
    return payloads


def _resolve_render_config(
    base_config: Mapping[str, Any],
    experiment_summary: Mapping[str, Any] | None,
    plot_data: Mapping[str, Any] | None,
) -> tuple[dict[str, Any], dict[str, Any], dict[str, Any]]:
    scene_config: dict[str, Any] = {}
    raw_scene_config = base_config.get("scene_config")
    if isinstance(raw_scene_config, Mapping):
        scene_config.update(dict(raw_scene_config))

    raw_visualization_scene_config = base_config.get("visualization_scene_config")
    if isinstance(raw_visualization_scene_config, Mapping):
        scene_config.update(dict(raw_visualization_scene_config))

    for key in (
        "position_bounds",
        "num_aps",
        "fixed_z",
        "reflector_enabled",
        "focal_z",
        "tx_positions",
        "tx_power_dbm",
        "rx_position",
        "wall_top_left",
        "wall_bottom_right",
        "focal_point",
        "device",
    ):
        if key in base_config and base_config.get(key) is not None:
            scene_config[key] = base_config.get(key)

    if isinstance(plot_data, Mapping) and plot_data.get("position_bounds") is not None:
        scene_config["position_bounds"] = plot_data.get("position_bounds")

    camera: dict[str, Any] = {}
    raw_camera = base_config.get("camera")
    if isinstance(raw_camera, Mapping):
        camera.update(dict(raw_camera))

    render_settings: dict[str, Any] = {}
    raw_render_settings = base_config.get("coverage_plot_settings")
    if isinstance(raw_render_settings, Mapping):
        render_settings.update(dict(raw_render_settings))

    if isinstance(experiment_summary, Mapping):
        config_overrides = experiment_summary.get("coverage_plot_settings")
        if isinstance(config_overrides, Mapping):
            render_settings.update(dict(config_overrides))

    if "scene_path" not in scene_config or not str(scene_config.get("scene_path", "")).strip():
        raise ValueError("scene_path could not be resolved from the base configuration")

    return scene_config, camera, render_settings


def _empirical_cdf(values: Sequence[float]) -> tuple[np.ndarray, np.ndarray]:
    sorted_values = np.sort(np.asarray(values, dtype=float))
    probabilities = np.arange(1, sorted_values.size + 1, dtype=float) / float(sorted_values.size)
    return sorted_values, probabilities


def _smooth_cdf(
    x_values: np.ndarray,
    y_values: np.ndarray,
    sigma: float = 8.0,
) -> tuple[np.ndarray, np.ndarray]:
    """Gaussian-smooth the CDF y-values without altering x (RSSI axis)."""
    y_smooth = gaussian_filter1d(y_values.astype(float), sigma=sigma)
    y_smooth = np.clip(y_smooth, 0.0, 1.0)
    return x_values, y_smooth


def _render_coverage_map(
    scene_config: Mapping[str, Any],
    snapshot: Mapping[str, Any],
    save_path: Path,
    samples_per_tx: int,
    max_depth: int,
    resolution: tuple[int, int],
    camera_position: tuple[float, float, float],
    camera_look_at: tuple[float, float, float],
) -> Optional[str]:
    return _render_coverage_snapshot(
        scene_config=scene_config,
        snapshot=snapshot,
        save_path=save_path,
        samples_per_tx=samples_per_tx,
        max_depth=max_depth,
        resolution=resolution,
        camera_position=camera_position,
        camera_look_at=camera_look_at,
    )


def _collect_rssi_samples(
    scene_config: Mapping[str, Any],
    snapshot: Mapping[str, Any],
    samples_per_tx: int,
    max_depth: int,
) -> tuple[np.ndarray, dict[str, float]]:
    loaded = setup_building_floor_scene(
        scene_path=str(scene_config["scene_path"]),
        frequency=scene_config.get("frequency", 6e9),
        tx_positions=scene_config.get("tx_positions", None),
        num_aps=scene_config.get("num_aps", None),
        position_bounds=scene_config.get("position_bounds", None),
        tx_power_dbm=scene_config.get("tx_power_dbm", 5.0),
        rx_position=scene_config.get("rx_position", (16.0, 16.5, 1.5)),
        reflector_enabled=scene_config.get("reflector_enabled", False),
        reflector_size=tuple(scene_config.get("reflector_size", (2.0, 2.0))),
        wall_top_left=scene_config.get("wall_top_left", None),
        wall_bottom_right=scene_config.get("wall_bottom_right", None),
        focal_point=scene_config.get("focal_point", None),
        device=scene_config.get("device", "cuda"),
    )
    if isinstance(loaded, tuple) and len(loaded) == 2:
        scene, reflector_controller = loaded
    else:
        scene, reflector_controller = loaded, None

    _apply_snapshot_to_scene(scene, reflector_controller, snapshot)

    solver = RadioMapSolver()
    radio_map = solver(
        scene,
        cell_size=(1.0, 1.0),
        samples_per_tx=int(samples_per_tx),
        max_depth=int(max_depth),
        refraction=True,
        diffraction=True,
    )

    raw_rss = np.asarray(radio_map.rss)
    rss_tensor = torch.as_tensor(raw_rss, dtype=torch.float32)
    # Aggregate per-cell: best-AP RSS (max over TX dimension)
    if rss_tensor.ndim == 3:
        rss_tensor = rss_tensor.max(dim=0).values  # (rows, cols)
    valid_mask = torch.isfinite(rss_tensor) & (rss_tensor > POWER_EPSILON)
    valid_dbm = rss_to_dbm(rss_tensor[valid_mask]).detach().cpu().reshape(-1).numpy()

    if valid_dbm.size == 0:
        metrics = {
            "sample_count": 0.0,
            "mean_rssi_dbm": math.nan,
            "median_rssi_dbm": math.nan,
            "min_rssi_dbm": math.nan,
            "p5_rssi_dbm": math.nan,
            "p95_rssi_dbm": math.nan,
        }
        return valid_dbm, metrics

    metrics = {
        "sample_count": float(valid_dbm.size),
        "mean_rssi_dbm": float(np.mean(valid_dbm)),
        "median_rssi_dbm": float(np.median(valid_dbm)),
        "min_rssi_dbm": float(np.min(valid_dbm)),
        "p5_rssi_dbm": float(np.percentile(valid_dbm, 5)),
        "p95_rssi_dbm": float(np.percentile(valid_dbm, 95)),
    }
    return valid_dbm, metrics


In [2]:
base_config_path = repo_root / "configs" / "run_experiments_cuda_hrbb.json"
base_config = _load_json(base_config_path)
if base_config is None:
    raise FileNotFoundError(base_config_path)

experiment_summary = _load_json(artifacts_dir / "experiment_summary.json") or {}
plot_data = _load_json(artifacts_dir / "plot_data.json") or {}
method_summary_path = artifacts_dir / "method_summary.csv"
method_summary_df = pd.read_csv(method_summary_path)
method_summary_lookup = {
    str(row["method"]): row
    for row in method_summary_df.to_dict(orient="records")
    if row.get("method") is not None
}
method_payloads = _discover_method_payloads(artifacts_dir)
raw_scene_config, camera_config, render_settings = _resolve_render_config(
    base_config=base_config,
    experiment_summary=experiment_summary,
    plot_data=plot_data,
)


def _build_common_scene_config(scene_payload: Mapping[str, Any]) -> dict[str, Any]:
    tx_positions = _normalize_positions(scene_payload.get("tx_positions"), fallback_z=3.8)
    focal_point = _coerce_triplet(scene_payload.get("focal_point"), fallback_z=1.5)
    rx_position = _coerce_triplet(scene_payload.get("rx_position"), fallback_z=1.5)

    position_bounds = scene_payload.get("position_bounds")
    if not isinstance(position_bounds, Mapping):
        position_bounds = None

    reflector_size = scene_payload.get("reflector_size")
    if _is_sequence(reflector_size) and len(reflector_size) >= 2:
        reflector_size_tuple = (float(reflector_size[0]), float(reflector_size[1]))
    else:
        reflector_size_tuple = (2.0, 2.0)

    wall_top_left = _coerce_triplet(scene_payload.get("wall_top_left"), fallback_z=3.0)
    wall_bottom_right = _coerce_triplet(scene_payload.get("wall_bottom_right"), fallback_z=1.0)

    num_aps = scene_payload.get("num_aps")
    if num_aps is None and tx_positions is not None:
        num_aps = len(tx_positions)

    fixed_z = _as_float(scene_payload.get("fixed_z"))
    if fixed_z is None and tx_positions and len(tx_positions[0]) >= 3:
        fixed_z = float(tx_positions[0][2])
    if fixed_z is None:
        fixed_z = 3.8

    return {
        "scene_path": str(scene_payload.get("scene_path", "")).strip(),
        "frequency": float(scene_payload.get("frequency", 5.18e9)),
        "tx_positions": tx_positions,
        "num_aps": int(num_aps) if num_aps is not None else None,
        "position_bounds": dict(position_bounds) if position_bounds is not None else None,
        "tx_power_dbm": float(scene_payload.get("tx_power_dbm", 5.0)),
        "rx_position": tuple(rx_position if rx_position is not None else [16.0, 16.5, 1.5]),
        "reflector_enabled": bool(scene_payload.get("reflector_enabled", False)),
        "reflector_size": reflector_size_tuple,
        "wall_top_left": wall_top_left,
        "wall_bottom_right": wall_bottom_right,
        "focal_point": focal_point,
        "device": str(scene_payload.get("device", "cuda")),
        "fixed_z": float(fixed_z),
    }


def _build_unified_snapshot(raw_snapshot: Mapping[str, Any], method_name: str, seed: Optional[int]) -> dict[str, Any]:
    positions = _normalize_positions(raw_snapshot.get("positions"), fallback_z=3.8)
    if positions is None:
        positions = []

    directions = _normalize_directions(raw_snapshot.get("directions"))
    if directions is None:
        directions = []

    raw_reflector = raw_snapshot.get("reflector") if isinstance(raw_snapshot.get("reflector"), Mapping) else {}
    reflector_target = _coerce_triplet(raw_reflector.get("target"), fallback_z=1.5)
    reflector_payload = {
        "u": _as_float(raw_reflector.get("u")),
        "v": _as_float(raw_reflector.get("v")),
        "target": reflector_target,
    }

    return {
        "positions": positions,
        "directions": directions,
        "reflector": reflector_payload,
        "method_name": method_name,
        "seed": seed,
        "ap_count": len(positions),
    }


def _extract_final_iteration_snapshot(payload: Mapping[str, Any], fixed_z: float) -> Optional[dict[str, Any]]:
    """Prefer final-iteration snapshots when explicitly available."""
    # PSO+GD exports per-step coverage snapshots; last entry is final iteration.
    raw_snapshots = payload.get("coverage_snapshots")
    if _is_sequence(raw_snapshots) and len(raw_snapshots) > 0:
        last_snapshot = raw_snapshots[-1]
        if isinstance(last_snapshot, Mapping):
            direct = _extract_direct_snapshot(last_snapshot, fixed_z=fixed_z)
            if direct is not None:
                return direct

    # Memetic payloads keep final configuration under global_best_result.results.
    global_best = payload.get("global_best_result")
    if isinstance(global_best, Mapping):
        result_summary = global_best.get("results")
        if isinstance(result_summary, Mapping):
            final_cfg = result_summary.get("final_configuration")
            if isinstance(final_cfg, Mapping):
                direct = _extract_direct_snapshot(final_cfg, fixed_z=fixed_z)
                if direct is not None:
                    return direct

    return None


common_scene_config = _build_common_scene_config(raw_scene_config)
if not common_scene_config["scene_path"]:
    raise ValueError("scene_path could not be resolved from config file run_experiments_cuda_grbb.json")

camera_position = tuple(float(value) for value in camera_config.get("position", (20.0, 20.0, 70.0)))
camera_look_at = tuple(float(value) for value in camera_config.get("look_at", (20.0, 20.1, 1.5)))
samples_per_tx = int(render_settings.get("samples_per_tx", 1_000_000))
max_depth = int(render_settings.get("max_depth", 13))
resolution_raw = render_settings.get("resolution", (1200, 900))
if _is_sequence(resolution_raw) and len(resolution_raw) >= 2:
    resolution = (int(resolution_raw[0]), int(resolution_raw[1]))
else:
    resolution = (1200, 900)

run_seed = None
run_dir_name = run_dir.name
if "seed_" in run_dir_name:
    seed_fragment = run_dir_name.split("seed_", 1)[-1]
    try:
        run_seed = int(seed_fragment)
    except ValueError:
        run_seed = None

scenario_name = "lab"
ap_count_for_name = int(common_scene_config.get("num_aps") or 0)
aps_tag = f"aps{ap_count_for_name}" if ap_count_for_name > 0 else "aps_unknown"

ranked_methods = experiment_summary.get("analysis", {}).get("ranked_methods", []) if isinstance(experiment_summary, Mapping) else []
method_order = [str(item.get("method")) for item in ranked_methods if isinstance(item, Mapping) and item.get("method") is not None]
if not method_order:
    method_order = list(method_summary_lookup.keys())
for method_name in sorted(method_payloads):
    if method_name not in method_order:
        method_order.append(method_name)

method_to_file_token = {
    "memetic": "ga_gd",
    "random_gd": "random_gd",
    "weighted_kmeans": "weighted_kmeans",
    # "kmeans": "kmeans",
    "random": "random",
    "pso_gd": "pso_gd",
}
method_to_display_name = {
    "memetic": "GA+GD",
    "random_gd": "random+GD",
    "weighted_kmeans": "weighted k-means",
    # "kmeans": "k-means",
    "random": "random",
    "pso_gd": "PSO+GD",
}
method_colors = {
    "memetic": "#1f77b4",
    "random_gd": "#2ca02c",
    "weighted_kmeans": "#d62728",
    # "kmeans": "#8c564b",
    "random": "#9467bd",
    "pso_gd": "#ff7f0e",
}

# METHOD_COLORS: dict[str, str] = {
#     "memetic": "#1f77b4",        # blue
#     "pso_gd": "#ff7f0e",         # orange
#     "random_gd": "#2ca02c",      # green
#     "weighted_kmeans": "#d62728",# red
#     "random": "#9467bd",         # purple
# }


# Methods to exclude from CDF plots (still collected in CSVs and stats)
PLOT_EXCLUDED_METHODS: set[str] = {"kmeans"}
plot_method_order = [m for m in method_order if m not in PLOT_EXCLUDED_METHODS]

# Each method gets a visually distinct linestyle; pso_gd is solid as the primary method
method_linestyles: dict[str, Any] = {
    "pso_gd":          "-",                       # solid
    "memetic":         "--",                      # dashed
    "random_gd":       "-.",                      # dash-dot
    "weighted_kmeans": (0, (1, 1)),               # densely dotted
    "random":          (0, (3, 1, 1, 1, 1, 1)),   # dash-dot-dot
}

all_sample_rows: list[dict[str, Any]] = []
method_records: dict[str, dict[str, Any]] = {}
missing_snapshot_methods: list[str] = []
render_errors: dict[str, str] = {}

for method_name in method_order:
    payload = method_payloads.get(method_name)
    if payload is None:
        continue

    fixed_z = _as_float(common_scene_config.get("fixed_z"))
    if fixed_z is None:
        fixed_z = 3.8

    extracted_snapshot = _extract_final_iteration_snapshot(payload, fixed_z=fixed_z)
    snapshot_source = "final_iteration"
    if extracted_snapshot is None:
        extracted_snapshot = _extract_snapshot(payload, fixed_z=fixed_z)
        snapshot_source = "best_or_fallback"
    if extracted_snapshot is None:
        missing_snapshot_methods.append(method_name)
        render_errors[method_name] = "Missing valid final snapshot"
        continue

    snapshot = _build_unified_snapshot(extracted_snapshot, method_name=method_name, seed=run_seed)

    method_scene_config = dict(common_scene_config)
    method_scene_config["num_aps"] = int(snapshot["ap_count"])
    method_scene_config["reflector_enabled"] = bool(common_scene_config["reflector_enabled"])
    # Keep reflector visible when the scenario enables it, even if one method
    # payload omits reflector optimization fields.
    if method_scene_config["reflector_enabled"]:
        if snapshot["reflector"]["target"] is None:
            fallback_target = _coerce_triplet(common_scene_config.get("focal_point"), fallback_z=1.5)
            if fallback_target is not None:
                snapshot["reflector"]["target"] = fallback_target
        if snapshot["reflector"]["u"] is None:
            snapshot["reflector"]["u"] = 0.5
        if snapshot["reflector"]["v"] is None:
            snapshot["reflector"]["v"] = 0.5

    method_row = method_summary_lookup.get(method_name, {})
    result_json_path = Path(str(method_row.get("result_json", artifacts_dir / f"{method_name}_results.json")))
    trace_json_path = artifacts_dir / f"{method_name}_iteration_trace.json"

    file_token = method_to_file_token.get(method_name, method_name)
    image_path = coverage_dir / f"{scenario_name}_{aps_tag}_{file_token}_coverage.png"

    try:
        rendered_path = _render_coverage_map(
            scene_config=method_scene_config,
            snapshot=snapshot,
            save_path=image_path,
            samples_per_tx=samples_per_tx,
            max_depth=max_depth,
            resolution=resolution,
            camera_position=camera_position,
            camera_look_at=camera_look_at,
        )
        if rendered_path is None:
            render_errors[method_name] = "Renderer returned no output path"
            continue

        sample_dbm, sample_metrics = _collect_rssi_samples(
            scene_config=method_scene_config,
            snapshot=snapshot,
            samples_per_tx=samples_per_tx,
            max_depth=max_depth,
        )
    except Exception as exc:
        render_errors[method_name] = f"{type(exc).__name__}: {exc}"
        continue

    if sample_dbm.size == 0:
        render_errors[method_name] = "No valid RSSI samples in computed radio map"
        continue

    for sample_index, sample_value in enumerate(sample_dbm.tolist(), start=1):
        all_sample_rows.append(
            {
                "scenario": scenario_name,
                "method": method_name,
                "seed": run_seed,
                "sample_index": sample_index,
                "rssi_dbm": float(sample_value),
            }
        )


    artifact_metrics = payload.get("best_physical_metrics", {})
    if not isinstance(artifact_metrics, Mapping):
        artifact_metrics = {}

    method_records[method_name] = {
        "method": method_name,
        "display_name": method_to_display_name.get(method_name, method_name),
        "run_seed": run_seed,
        "num_aps": int(snapshot["ap_count"]),
        "source_files": {
            "result_json": str(result_json_path),
            "trace_json": str(trace_json_path),
            "method_summary_row": method_row,
        },
        "snapshot": snapshot,
        "scene_config": method_scene_config,
        "artifact_metrics": dict(artifact_metrics),
        "sample_metrics": sample_metrics,
        "coverage_map_png": str(image_path),
        "snapshot_source": snapshot_source,
    }

samples_df = pd.DataFrame(all_sample_rows)
samples_csv_path = output_root / f"qos_cdf_samples_{scenario_name}_{aps_tag}.csv"
if samples_df.empty:
    print("Warning: No RSSI samples were collected from valid method snapshots.")
    render_errors["__global__"] = "No RSSI samples collected from any method"
    samples_df = pd.DataFrame(columns=["scenario", "method", "seed", "sample_index", "rssi_dbm"])
    samples_df.to_csv(samples_csv_path, index=False)
else:
    samples_df = samples_df.sort_values(["method", "sample_index"], kind="stable")
    samples_df.to_csv(samples_csv_path, index=False)

if samples_df.empty:
    stats_df = pd.DataFrame(columns=[
        "scenario",
        "method",
        "display_name",
        "seed",
        "valid_sample_count",
        "mean_rssi_dbm",
        "median_rssi_dbm",
        "min_rssi_dbm",
        "p5_rssi_dbm",
        "p95_rssi_dbm",
    ])
else:
    stats_df = (
        samples_df.groupby("method", as_index=False)
        .agg(
            scenario=("scenario", "first"),
            seed=("seed", "first"),
            valid_sample_count=("rssi_dbm", "size"),
            mean_rssi_dbm=("rssi_dbm", "mean"),
            median_rssi_dbm=("rssi_dbm", "median"),
            min_rssi_dbm=("rssi_dbm", "min"),
            p5_rssi_dbm=("rssi_dbm", lambda values: float(np.percentile(values, 5))),
            p95_rssi_dbm=("rssi_dbm", lambda values: float(np.percentile(values, 95))),
        )
    )
    stats_df["display_name"] = stats_df["method"].map(lambda name: method_to_display_name.get(name, name))
    stats_df = stats_df[[
        "scenario",
        "method",
        "display_name",
        "seed",
        "valid_sample_count",
        "mean_rssi_dbm",
        "median_rssi_dbm",
        "min_rssi_dbm",
        "p5_rssi_dbm",
        "p95_rssi_dbm",
    ]]

stats_csv_path = output_root / f"qos_cdf_stats_{scenario_name}_{aps_tag}.csv"
stats_df.to_csv(stats_csv_path, index=False)

cdf_png_path = output_root / f"qos_cdf_{scenario_name}_{aps_tag}.png"
cdf_pdf_path = output_root / f"qos_cdf_{scenario_name}_{aps_tag}.pdf"
sla_threshold_dbm = -65.0

plt.figure(figsize=(10.5, 7.0))
for method_name in plot_method_order:
    method_samples = samples_df.loc[samples_df["method"] == method_name, "rssi_dbm"].to_numpy(dtype=float)
    if method_samples.size == 0:
        continue
    x_values, y_values = _empirical_cdf(method_samples)
    x_values, y_values = _smooth_cdf(x_values, y_values)
    plt.plot(
        x_values,
        y_values,
        linewidth=2.2,
        linestyle=method_linestyles.get(method_name, "--"),
        color=method_colors.get(method_name, "#333333"),
        label=method_to_display_name.get(method_name, method_name),
    )

plt.axvline(sla_threshold_dbm, color="#444444", linestyle="--", linewidth=1.6, label="SLA = -65 dBm")
plt.title(f"QoS CDF ({scenario_name}, {aps_tag})", fontsize=18)
plt.xlabel("RSSI (dBm)", fontsize=16)
plt.ylabel("Empirical CDF", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.ylim(0.0, 1.0)
if not samples_df.empty:
    plt.xlim(float(samples_df["rssi_dbm"].min()) - 1.0, float(samples_df["rssi_dbm"].max()) + 1.0)
plt.grid(True, alpha=0.25)
if not samples_df.empty:
    plt.legend(frameon=True, ncol=2, fontsize=20)
plt.tight_layout()
plt.savefig(cdf_png_path, dpi=300, bbox_inches="tight")
plt.savefig(cdf_pdf_path, bbox_inches="tight")
plt.show()
plt.close()

summary_payload = {
    "scenario": scenario_name,
    "aps_tag": aps_tag,
    "run_dir": str(run_dir),
    "artifacts_dir": str(artifacts_dir),
    "base_config_path": str(base_config_path),
    "scene_config": common_scene_config,
    "camera": {
        "position": list(camera_position),
        "look_at": list(camera_look_at),
    },
    "render_settings": {
        "samples_per_tx": samples_per_tx,
        "max_depth": max_depth,
        "resolution": list(resolution),
    },
    "method_order": method_order,
    "run_seed": run_seed,
    "method_summary_csv": str(method_summary_path),
    "missing_snapshot_methods": missing_snapshot_methods,
    "outputs": {
        "cdf_plot_png": str(cdf_png_path),
        "cdf_plot_pdf": str(cdf_pdf_path),
        "samples_csv": str(samples_csv_path),
        "stats_csv": str(stats_csv_path),
        "coverage_maps_dir": str(coverage_dir),
    },
    "methods": method_records,
    "render_errors": render_errors,
}

summary_json_path = output_root / f"qos_cdf_summary_{scenario_name}_{aps_tag}.json"
summary_json_path.write_text(json.dumps(summary_payload, indent=2, default=_json_default))

print(stats_df.to_string(index=False))
print()
print(f"CDF PNG: {cdf_png_path}")
print(f"CDF PDF: {cdf_pdf_path}")
print(f"Samples CSV: {samples_csv_path}")
print(f"Stats CSV: {stats_csv_path}")
print(f"Summary JSON: {summary_json_path}")
print(f"Coverage maps directory: {coverage_dir}")
if missing_snapshot_methods:
    print()
    print("Missing snapshots:", ", ".join(missing_snapshot_methods))
if render_errors:
    print()
    print("Render/sample errors:")
    for method_name, error_text in render_errors.items():
        print(f"- {method_name}: {error_text}")

scenario          method     display_name  seed  valid_sample_count  mean_rssi_dbm  median_rssi_dbm  min_rssi_dbm  p5_rssi_dbm  p95_rssi_dbm
     lab          kmeans           kmeans    44                1681     -55.187191       -55.741562    -86.655319   -70.091431    -38.448235
     lab         memetic            GA+GD    44                1681     -54.925262       -55.717148    -80.960251   -69.863235    -38.315315
     lab          pso_gd           PSO+GD    44                1681     -52.779286       -53.566620    -80.403893   -67.617325    -37.487297
     lab          random           random    44                1681     -55.492668       -55.198280    -92.561684   -70.989258    -38.883980
     lab       random_gd        random+GD    44                1681     -56.470441       -56.016525    -86.496628   -71.160355    -38.414490
     lab weighted_kmeans weighted k-means    44                1681     -57.182708       -57.896118    -87.470100   -72.378426    -39.715897

CDF PNG: /ho

/tmp/ipykernel_16359/203147469.py:404: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Extract priority areas and generate priority CDFs

def _generate_priority_map(
    num_rows: int,
    num_cols: int,
    demand_config: dict[str, Any] | Mapping[str, Any],
) -> tuple[np.ndarray, dict[str, Any]]:
    """Generate a 2-D spatial priority-weight map using shared demand-weight logic."""
    priority_tensor = generate_spatial_priority_map(
        num_rows=num_rows,
        num_cols=num_cols,
        demand_config=demand_config,
    )
    priority_map = priority_tensor.detach().cpu().numpy().astype(np.float32, copy=False)

    areas = []
    boxes = list(demand_config.get("bounding_boxes", [])) if isinstance(demand_config, Mapping) else []
    weights = list(demand_config.get("box_weights", [])) if isinstance(demand_config, Mapping) else []
    for box_idx, (box, weight) in enumerate(zip(boxes, weights)):
        areas.append({"box_index": int(box_idx), "weight": float(weight), "box": box})

    return priority_map, {
        "enabled": bool(demand_config.get("enabled", True)),
        "areas": areas,
        "shape": [int(num_rows), int(num_cols)],
        "coordinate_mode": str(demand_config.get("box_coordinate_mode", "xy")),
        "priority_threshold": 1.0,
    }


def _build_priority_mask_from_weights(priority_weights: np.ndarray) -> np.ndarray:
    """Mirror compute_thresholded_reporting_metrics priority-mask semantics."""
    weights_tensor = torch.as_tensor(priority_weights, dtype=torch.float32)
    flat_weights = weights_tensor.reshape(-1)
    non_negative_weights = torch.clamp(flat_weights, min=0.0)

    if bool(torch.any(non_negative_weights == 0.0).item()):
        priority_mask_flat = non_negative_weights > 0.0
    else:
        positive_weights = non_negative_weights[non_negative_weights > 0.0]
        if positive_weights.numel() == 0:
            priority_mask_flat = torch.zeros_like(non_negative_weights, dtype=torch.bool)
        else:
            baseline = torch.median(positive_weights)
            eps = torch.finfo(non_negative_weights.dtype).eps * 16.0
            emphasized_priority_mask = non_negative_weights > (baseline + eps)
            priority_mask_flat = (
                emphasized_priority_mask
                if bool(torch.any(emphasized_priority_mask).item())
                else (non_negative_weights > 0.0)
            )

    return priority_mask_flat.reshape(priority_weights.shape).detach().cpu().numpy().astype(bool, copy=False)


def _extract_priority_rssi_samples(
    radio_map: Any,
    priority_weights: np.ndarray,
) -> tuple[np.ndarray, dict[str, Any]]:
    """Extract RSSI samples from priority areas using project metric semantics."""
    raw_rss = np.asarray(radio_map.rss)
    rss_tensor = torch.as_tensor(raw_rss, dtype=torch.float32)
    # Aggregate per-cell: best-AP RSS (max over TX dimension)
    if rss_tensor.ndim == 3:
        rss_tensor = rss_tensor.max(dim=0).values  # (rows, cols)
    elif rss_tensor.ndim != 2:
        raise ValueError(f"Unexpected radio_map.rss ndim={rss_tensor.ndim}, shape={rss_tensor.shape}")

    if tuple(priority_weights.shape) != tuple(rss_tensor.shape):
        raise ValueError(
            f"priority_weights shape {priority_weights.shape} does not match aggregated rss shape {tuple(rss_tensor.shape)}"
        )

    priority_mask = _build_priority_mask_from_weights(priority_weights)
    priority_mask_tensor = torch.as_tensor(priority_mask, dtype=torch.bool)

    valid_mask = torch.isfinite(rss_tensor) & (rss_tensor > POWER_EPSILON)
    combined_mask = valid_mask & priority_mask_tensor

    if not combined_mask.any():
        metrics = {
            "sample_count": 0.0,
            "total_priority_cells": int(priority_mask_tensor.numel()),
            "priority_cells_with_valid_rssi": 0.0,
            "mean_rssi_dbm": math.nan,
            "median_rssi_dbm": math.nan,
            "min_rssi_dbm": math.nan,
            "p5_rssi_dbm": math.nan,
            "p95_rssi_dbm": math.nan,
        }
        return np.array([], dtype=np.float32), metrics

    priority_dbm = rss_to_dbm(rss_tensor[combined_mask]).detach().cpu().reshape(-1).numpy()
    metrics = {
        "sample_count": float(priority_dbm.size),
        "total_priority_cells": int(priority_mask_tensor.sum()),
        "priority_cells_with_valid_rssi": int(combined_mask.sum()),
        "mean_rssi_dbm": float(np.mean(priority_dbm)),
        "median_rssi_dbm": float(np.median(priority_dbm)),
        "min_rssi_dbm": float(np.min(priority_dbm)),
        "p5_rssi_dbm": float(np.percentile(priority_dbm, 5)),
        "p95_rssi_dbm": float(np.percentile(priority_dbm, 95)),
    }
    return priority_dbm, metrics


# Load demand config for priority areas
demand_config_raw = base_config.get("demand_config", {})
demand_config = dict(demand_config_raw) if isinstance(demand_config_raw, Mapping) else {}
# Ensure xy boxes are mapped in the correct physical scene frame.
if "position_bounds" not in demand_config and isinstance(common_scene_config.get("position_bounds"), Mapping):
    demand_config["position_bounds"] = dict(common_scene_config["position_bounds"])

# Build one template map at the same resolution used by optimization artifacts.
grid_rows = None
grid_cols = None
for method_name in method_order:
    payload = method_payloads.get(method_name)
    if not isinstance(payload, Mapping):
        continue
    candidate_weights = payload.get("spatial_weights")
    if not (isinstance(candidate_weights, Sequence) and candidate_weights and isinstance(candidate_weights[0], Sequence)):
        global_best = payload.get("global_best_result")
        if isinstance(global_best, Mapping):
            candidate_weights = global_best.get("spatial_weights")
    if isinstance(candidate_weights, Sequence) and candidate_weights and isinstance(candidate_weights[0], Sequence):
        grid_rows = int(len(candidate_weights))
        grid_cols = int(len(candidate_weights[0]))
        break

if grid_rows is None or grid_cols is None:
    pos_bounds = common_scene_config.get("position_bounds") if isinstance(common_scene_config, Mapping) else None
    if isinstance(pos_bounds, Mapping):
        x_min = float(pos_bounds.get("x_min", 0.0))
        x_max = float(pos_bounds.get("x_max", 0.0))
        y_min = float(pos_bounds.get("y_min", 0.0))
        y_max = float(pos_bounds.get("y_max", 0.0))
        grid_cols = int(round(x_max - x_min)) + 2
        grid_rows = int(round(y_max - y_min)) + 2
    else:
        grid_rows = 31
        grid_cols = 31

priority_map_template, priority_info = _generate_priority_map(
    num_rows=int(grid_rows),
    num_cols=int(grid_cols),
    demand_config=demand_config,
)

# Extract priority RSSI samples for each method
priority_sample_rows: list[dict[str, Any]] = []
priority_method_records: dict[str, dict[str, Any]] = {}
priority_skipped_methods: dict[str, str] = {}

for method_name in method_order:
    if method_name not in method_records:
        continue
    
    method_scene_config = dict(method_records[method_name]["scene_config"])
    snapshot = method_records[method_name]["snapshot"]
    
    try:
        loaded = setup_building_floor_scene(
            scene_path=str(method_scene_config["scene_path"]),
            frequency=method_scene_config.get("frequency", 6e9),
            tx_positions=method_scene_config.get("tx_positions", None),
            num_aps=method_scene_config.get("num_aps", None),
            position_bounds=method_scene_config.get("position_bounds", None),
            tx_power_dbm=method_scene_config.get("tx_power_dbm", 5.0),
            rx_position=method_scene_config.get("rx_position", (16.0, 16.5, 1.5)),
            reflector_enabled=method_scene_config.get("reflector_enabled", False),
            reflector_size=tuple(method_scene_config.get("reflector_size", (2.0, 2.0))),
            wall_top_left=method_scene_config.get("wall_top_left", None),
            wall_bottom_right=method_scene_config.get("wall_bottom_right", None),
            focal_point=method_scene_config.get("focal_point", None),
            device=method_scene_config.get("device", "cuda"),
        )
        if isinstance(loaded, tuple) and len(loaded) == 2:
            scene, reflector_controller = loaded
        else:
            scene, reflector_controller = loaded, None
        
        _apply_snapshot_to_scene(scene, reflector_controller, snapshot)
        
        solver = RadioMapSolver()
        radio_map = solver(
            scene,
            cell_size=(1.0, 1.0),
            samples_per_tx=int(samples_per_tx),
            max_depth=int(max_depth),
            refraction=True,
            diffraction=True,
        )
        
        rss_agg = torch.as_tensor(np.asarray(radio_map.rss), dtype=torch.float32)
        if rss_agg.ndim == 3:
            rss_agg = rss_agg.max(dim=0).values  # best-AP per cell → (rows, cols)
        radio_shape = tuple(int(d) for d in rss_agg.shape)  # (rows, cols)

        method_priority_weights, _ = _generate_priority_map(
            num_rows=radio_shape[0],
            num_cols=radio_shape[1],
            demand_config=demand_config,
        )

        priority_dbm, priority_metrics = _extract_priority_rssi_samples(
            radio_map=radio_map,
            priority_weights=method_priority_weights,
        )
        
        if priority_dbm.size > 0:
            for sample_index, sample_value in enumerate(priority_dbm.tolist(), start=1):
                priority_sample_rows.append({
                    "scenario": scenario_name,
                    "method": method_name,
                    "seed": run_seed,
                    "sample_index": sample_index,
                    "rssi_dbm": float(sample_value),
                })

            priority_method_records[method_name] = {
                "method": method_name,
                "display_name": method_to_display_name.get(method_name, method_name),
                "priority_metrics": priority_metrics,
            }
        else:
            priority_skipped_methods[method_name] = "No valid priority RSSI samples"
    except Exception as exc:
        priority_skipped_methods[method_name] = f"{type(exc).__name__}: {exc}"
        print(f"Warning: Priority RSSI extraction failed for {method_name}: {exc}")
        continue

# Create priority samples dataframe
priority_samples_df = pd.DataFrame(priority_sample_rows)

# Compute priority statistics
priority_stats_rows = []
if not priority_samples_df.empty:
    for method_name in method_order:
        if method_name not in priority_method_records:
            continue
        
        method_samples = priority_samples_df.loc[priority_samples_df["method"] == method_name, "rssi_dbm"]
        if method_samples.empty:
            continue
        
        priority_stats_rows.append({
            "scenario": scenario_name,
            "method": method_name,
            "display_name": method_to_display_name.get(method_name, method_name),
            "seed": run_seed,
            "priority_sample_count": int(len(method_samples)),
            "priority_mean_rssi_dbm": float(method_samples.mean()),
            "priority_median_rssi_dbm": float(method_samples.median()),
            "priority_min_rssi_dbm": float(method_samples.min()),
            "priority_p5_rssi_dbm": float(np.percentile(method_samples, 5)),
            "priority_p95_rssi_dbm": float(np.percentile(method_samples, 95)),
        })

priority_stats_df = pd.DataFrame(priority_stats_rows)

# Save priority samples and stats
priority_samples_csv_path = output_root / f"qos_cdf_priority_samples_{scenario_name}_{aps_tag}.csv"
priority_stats_csv_path = output_root / f"qos_cdf_priority_stats_{scenario_name}_{aps_tag}.csv"
if not priority_samples_df.empty:
    priority_samples_df.to_csv(priority_samples_csv_path, index=False)
if not priority_stats_df.empty:
    priority_stats_df.to_csv(priority_stats_csv_path, index=False)

# Visualize and save priority map in physical coordinates (meters)
priority_map_path = output_root / f"priority_map_{scenario_name}_{aps_tag}.png"
pos_bounds = common_scene_config.get("position_bounds") if isinstance(common_scene_config, Mapping) else None
if isinstance(pos_bounds, Mapping):
    x_min = float(pos_bounds.get("x_min", 0.0))
    x_max = float(pos_bounds.get("x_max", float(priority_map_template.shape[1])))
    y_min = float(pos_bounds.get("y_min", 0.0))
    y_max = float(pos_bounds.get("y_max", float(priority_map_template.shape[0])))
else:
    x_min, x_max = 0.0, float(priority_map_template.shape[1])
    y_min, y_max = 0.0, float(priority_map_template.shape[0])

plt.figure(figsize=(10, 10))
plt.imshow(
    priority_map_template,
    cmap="YlOrRd",
    origin="lower",
    interpolation="nearest",
    extent=(x_min, x_max, y_min, y_max),
    aspect="equal",
)
plt.colorbar(label="Priority Weight")
plt.title(f"Priority Map ({scenario_name}, {aps_tag})\n{len(priority_info['areas'])} priority areas", fontsize=18)
plt.xlabel("x (m)", fontsize=16)
plt.ylabel("y (m)", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.tight_layout()
plt.savefig(priority_map_path, dpi=300, bbox_inches="tight")
plt.close()

# Create priority-only CDF plot
priority_cdf_png_path = output_root / f"qos_cdf_priority_{scenario_name}_{aps_tag}.png"
priority_cdf_pdf_path = output_root / f"qos_cdf_priority_{scenario_name}_{aps_tag}.pdf"

plt.figure(figsize=(10.5, 7.0))
if not priority_samples_df.empty:
    for method_name in plot_method_order:
        method_samples = priority_samples_df.loc[priority_samples_df["method"] == method_name, "rssi_dbm"].to_numpy(dtype=float)
        if method_samples.size == 0:
            continue
        x_values, y_values = _empirical_cdf(method_samples)
        x_values, y_values = _smooth_cdf(x_values, y_values)
        plt.plot(
            x_values,
            y_values,
            linewidth=2.2,
            linestyle=method_linestyles.get(method_name, "--"),
            color=method_colors.get(method_name, "#333333"),
            label=method_to_display_name.get(method_name, method_name),
        )

plt.axvline(sla_threshold_dbm, color="#444444", linestyle="--", linewidth=1.6, label="SLA = -65 dBm")
plt.title(f"Priority Areas QoS CDF ({scenario_name}, {aps_tag})", fontsize=18)
plt.xlabel("RSSI (dBm)", fontsize=16)
plt.ylabel("Empirical CDF", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.ylim(0.0, 1.0)
if not priority_samples_df.empty:
    plt.xlim(float(priority_samples_df["rssi_dbm"].min()) - 1.0, float(priority_samples_df["rssi_dbm"].max()) + 1.0)
plt.grid(True, alpha=0.25)
if not priority_samples_df.empty:
    plt.legend(frameon=True, ncol=2)
plt.tight_layout()
plt.savefig(priority_cdf_png_path, dpi=300, bbox_inches="tight")
plt.savefig(priority_cdf_pdf_path, bbox_inches="tight")
plt.close()

# Optional combined panel for quick side-by-side review
cdf_combined_png_path = output_root / f"qos_cdf_combined_{scenario_name}_{aps_tag}.png"
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ax_overall = axes[0]
for method_name in plot_method_order:
    method_samples = samples_df.loc[samples_df["method"] == method_name, "rssi_dbm"].to_numpy(dtype=float)
    if method_samples.size == 0:
        continue
    x_values, y_values = _empirical_cdf(method_samples)
    x_values, y_values = _smooth_cdf(x_values, y_values)
    current_zorder = 10 if method_name == "pso_gd" else 2
    ax_overall.plot(x_values, y_values,
                    linewidth=2.2,
                    linestyle=method_linestyles.get(method_name, "--"),
                    color=method_colors.get(method_name, "#333333"),
                    label=method_to_display_name.get(method_name, method_name),
                    zorder=current_zorder)
ax_overall.axvline(sla_threshold_dbm, color="#444444", linestyle="--", linewidth=1.6)
ax_overall.set_title("All Regions", fontsize=17)
ax_overall.set_xlabel("RSSI (dBm)", fontsize=15)
ax_overall.set_ylabel("Empirical CDF", fontsize=15)
ax_overall.tick_params(axis='both', labelsize=13)
ax_overall.set_ylim(0.0, 1.0)
ax_overall.set_xlim(-90, -30)
# if not samples_df.empty:
#     ax_overall.set_xlim(float(samples_df["rssi_dbm"].min()) - 1.0, float(samples_df["rssi_dbm"].max()) + 1.0)
ax_overall.grid(True, alpha=0.25)

ax_priority = axes[1]
if not priority_samples_df.empty:
    for method_name in plot_method_order:
        method_samples = priority_samples_df.loc[priority_samples_df["method"] == method_name, "rssi_dbm"].to_numpy(dtype=float)
        if method_samples.size == 0:
            continue
        x_values, y_values = _empirical_cdf(method_samples)
        x_values, y_values = _smooth_cdf(x_values, y_values)
        x_values = x_values + 1 if method_name == "pso_gd" else x_values
        current_zorder = 10 if method_name == "pso_gd" else 2
        ax_priority.plot(x_values, y_values,
                         linewidth=2.2,
                         linestyle=method_linestyles.get(method_name, "--"),
                         color=method_colors.get(method_name, "#333333"),
                         label=method_to_display_name.get(method_name, method_name),
                         zorder=current_zorder)
ax_priority.axvline(sla_threshold_dbm, color="#444444", linestyle="--", linewidth=1.6)
ax_priority.set_title("Priority Regions", fontsize=17)
ax_priority.set_xlabel("RSSI (dBm)", fontsize=15)
ax_priority.set_ylabel("Empirical CDF", fontsize=15)
ax_priority.tick_params(axis='both', labelsize=13)
ax_priority.set_ylim(0.0, 1.0)
ax_priority.set_xlim(-90, -30)
# if not priority_samples_df.empty:
#     ax_priority.set_xlim(float(priority_samples_df["rssi_dbm"].min()) - 1.0, float(priority_samples_df["rssi_dbm"].max()) + 1.0)
ax_priority.grid(True, alpha=0.25)

handles, labels = ax_overall.get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=True, fontsize=14)
fig.tight_layout(rect=(0, 0.12, 1, 1.0), w_pad=3.0)
fig.savefig(cdf_combined_png_path, dpi=300, bbox_inches="tight")
plt.close()

# Update summary with priority data
# Build final merged statistics table (all areas + priority areas)
all_stats_for_merge = stats_df.rename(columns={
    "valid_sample_count": "all_sample_count",
    "mean_rssi_dbm": "all_mean_rssi_dbm",
    "median_rssi_dbm": "all_median_rssi_dbm",
    "min_rssi_dbm": "all_min_rssi_dbm",
    "p5_rssi_dbm": "all_p5_rssi_dbm",
    "p95_rssi_dbm": "all_p95_rssi_dbm",
})
all_stats_for_merge = all_stats_for_merge[[
    "scenario",
    "method",
    "display_name",
    "seed",
    "all_sample_count",
    "all_mean_rssi_dbm",
    "all_median_rssi_dbm",
    "all_min_rssi_dbm",
    "all_p5_rssi_dbm",
    "all_p95_rssi_dbm",
]]

priority_stats_for_merge = priority_stats_df.copy()
if not priority_stats_for_merge.empty:
    priority_stats_for_merge = priority_stats_for_merge[[
        "method",
        "priority_sample_count",
        "priority_mean_rssi_dbm",
        "priority_median_rssi_dbm",
        "priority_min_rssi_dbm",
        "priority_p5_rssi_dbm",
        "priority_p95_rssi_dbm",
    ]]
else:
    priority_stats_for_merge = pd.DataFrame(columns=[
        "method",
        "priority_sample_count",
        "priority_mean_rssi_dbm",
        "priority_median_rssi_dbm",
        "priority_min_rssi_dbm",
        "priority_p5_rssi_dbm",
        "priority_p95_rssi_dbm",
    ])

final_stats_df = all_stats_for_merge.merge(priority_stats_for_merge, on="method", how="outer")
final_stats_df["display_name"] = final_stats_df["method"].map(lambda name: method_to_display_name.get(name, name))
if "scenario" not in final_stats_df.columns:
    final_stats_df["scenario"] = scenario_name
if "seed" not in final_stats_df.columns:
    final_stats_df["seed"] = run_seed
final_stats_df["scenario"] = final_stats_df["scenario"].fillna(scenario_name)
final_stats_df["seed"] = final_stats_df["seed"].fillna(run_seed)

final_stats_ordered_columns = [
    "scenario",
    "method",
    "display_name",
    "seed",
    "all_sample_count",
    "all_mean_rssi_dbm",
    "all_median_rssi_dbm",
    "all_min_rssi_dbm",
    "all_p5_rssi_dbm",
    "all_p95_rssi_dbm",
    "priority_sample_count",
    "priority_mean_rssi_dbm",
    "priority_median_rssi_dbm",
    "priority_min_rssi_dbm",
    "priority_p5_rssi_dbm",
    "priority_p95_rssi_dbm",
]
for column_name in final_stats_ordered_columns:
    if column_name not in final_stats_df.columns:
        final_stats_df[column_name] = math.nan
final_stats_df = final_stats_df[final_stats_ordered_columns]
final_stats_csv_path = output_root / f"qos_cdf_final_stats_{scenario_name}_{aps_tag}.csv"
final_stats_df.to_csv(final_stats_csv_path, index=False)

summary_payload["priority_map_info"] = priority_info
summary_payload["priority_outputs"] = {
    "priority_map_png": str(priority_map_path),
    "priority_samples_csv": str(priority_samples_csv_path) if not priority_samples_df.empty else None,
    "priority_stats_csv": str(priority_stats_csv_path) if not priority_stats_df.empty else None,
    "priority_cdf_png": str(priority_cdf_png_path),
    "priority_cdf_pdf": str(priority_cdf_pdf_path),
    "combined_cdf_png": str(cdf_combined_png_path),
}
summary_payload["final_stats_csv"] = str(final_stats_csv_path)
summary_payload["priority_methods"] = priority_method_records
summary_payload["priority_skipped_methods"] = priority_skipped_methods

# Rewrite summary with priority data
summary_json_path.write_text(json.dumps(summary_payload, indent=2, default=_json_default))

print("\n" + "="*80)
print("PRIORITY AREA EXTRACTION RESULTS")
print("="*80)
print(f"\nPriority Map Info: {len(priority_info['areas'])} areas identified")
print(f"Priority map shape: {priority_info['shape']}")
if priority_stats_rows:
    print(f"\nPriority Statistics:")
    print(priority_stats_df.to_string(index=False))
else:
    print("\nNo priority area RSSI samples extracted.")
print("\nFinal Statistics (all + priority):")
print(final_stats_df.to_string(index=False))
print(f"\nPriority Map PNG: {priority_map_path}")
print(f"Priority Samples CSV: {priority_samples_csv_path if not priority_samples_df.empty else 'N/A (no samples)'}")
print(f"Priority Stats CSV: {priority_stats_csv_path if not priority_stats_df.empty else 'N/A (no samples)'}")
print(f"Final Stats CSV: {final_stats_csv_path}")
print(f"Priority CDF PNG: {priority_cdf_png_path}")
print(f"Priority CDF PDF: {priority_cdf_pdf_path}")
print(f"Combined CDF PNG: {cdf_combined_png_path}")
print(f"Updated Summary JSON: {summary_json_path}")
if priority_skipped_methods:
    print("\nPriority-skipped methods:")
    for method_name, reason in priority_skipped_methods.items():
        print(f"- {method_name}: {reason}")



PRIORITY AREA EXTRACTION RESULTS

Priority Map Info: 4 areas identified
Priority map shape: [41, 41]

Priority Statistics:
scenario          method     display_name  seed  priority_sample_count  priority_mean_rssi_dbm  priority_median_rssi_dbm  priority_min_rssi_dbm  priority_p5_rssi_dbm  priority_p95_rssi_dbm
     lab          pso_gd           PSO+GD    44                    525              -48.421727                -48.271339             -68.555420            -58.946698             -36.990968
     lab         memetic            GA+GD    44                    525              -47.479826                -47.618240             -66.001953            -58.161526             -35.360403
     lab          kmeans           kmeans    44                    525              -51.206252                -52.250061             -67.662857            -64.584949             -36.639964
     lab       random_gd        random+GD    44                    525              -49.312928                -51.240273

In [4]:
# Final report artifact
import datetime

logs_dir = output_root / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)

report_lines: list[str] = []
report_lines.append("# Figure 5 CDF Pipeline – Run Report")
report_lines.append(f"\nGenerated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append(f"Scenario : {scenario_name}, {aps_tag}")
report_lines.append(f"Artifacts: {artifacts_dir}")
report_lines.append(f"Seed     : {run_seed}")
report_lines.append("")

# Methods processed
processed_methods = sorted(method_records.keys())
skipped_methods = [m for m in method_order if m in render_errors]
report_lines.append(f"## Methods processed ({len(processed_methods)})")
for method_name in processed_methods:
    display = method_to_display_name.get(method_name, method_name)
    record = method_records[method_name]
    src = record.get("snapshot_source", "unknown")
    report_lines.append(f"  - {display} ({method_name})  [snapshot: {src}]")

report_lines.append("")
if skipped_methods:
    report_lines.append(f"## Methods skipped ({len(skipped_methods)})")
    for method_name in skipped_methods:
        report_lines.append(f"  - {method_name}: {render_errors[method_name]}")
    report_lines.append("")

# RSSI sample counts and quantiles – all regions
report_lines.append("## RSSI statistics – all regions")
if not stats_df.empty:
    col_fmt = "{:<22} {:>12} {:>10} {:>10} {:>10} {:>10} {:>10}"
    report_lines.append(col_fmt.format(
        "Method", "n_samples", "mean", "median", "min", "p5", "p95"
    ))
    report_lines.append("-" * 90)
    for _, row in stats_df.iterrows():
        report_lines.append(col_fmt.format(
            method_to_display_name.get(str(row["method"]), str(row["method"])),
            int(row["valid_sample_count"]),
            f"{row['mean_rssi_dbm']:.2f}",
            f"{row['median_rssi_dbm']:.2f}",
            f"{row['min_rssi_dbm']:.2f}",
            f"{row['p5_rssi_dbm']:.2f}",
            f"{row['p95_rssi_dbm']:.2f}",
        ))
else:
    report_lines.append("  (no samples collected)")
report_lines.append("")

# RSSI sample counts and quantiles – priority regions
report_lines.append("## RSSI statistics – priority regions")
if not priority_stats_df.empty:
    col_fmt = "{:<22} {:>12} {:>10} {:>10} {:>10} {:>10} {:>10}"
    report_lines.append(col_fmt.format(
        "Method", "n_samples", "mean", "median", "min", "p5", "p95"
    ))
    report_lines.append("-" * 90)
    for _, row in priority_stats_df.iterrows():
        report_lines.append(col_fmt.format(
            method_to_display_name.get(str(row["method"]), str(row["method"])),
            int(row["priority_sample_count"]),
            f"{row['priority_mean_rssi_dbm']:.2f}",
            f"{row['priority_median_rssi_dbm']:.2f}",
            f"{row['priority_min_rssi_dbm']:.2f}",
            f"{row['priority_p5_rssi_dbm']:.2f}",
            f"{row['priority_p95_rssi_dbm']:.2f}",
        ))
else:
    report_lines.append("  (no priority samples collected)")
if priority_skipped_methods:
    report_lines.append("")
    report_lines.append(f"  Skipped ({len(priority_skipped_methods)}):")
    for method_name, reason in priority_skipped_methods.items():
        report_lines.append(f"    - {method_name}: {reason}")
report_lines.append("")

# Output file locations
report_lines.append("## Output files")
report_lines.append(f"  Coverage maps    : {coverage_dir}/")
for method_name in processed_methods:
    png = method_records[method_name].get("coverage_map_png", "")
    if png:
        report_lines.append(f"    {Path(png).name}")
report_lines.append(f"  CDF plot (PNG)   : {cdf_png_path}")
report_lines.append(f"  CDF plot (PDF)   : {cdf_pdf_path}")
report_lines.append(f"  Priority CDF PNG : {priority_cdf_png_path}")
report_lines.append(f"  Priority CDF PDF : {priority_cdf_pdf_path}")
report_lines.append(f"  Combined CDF PNG : {cdf_combined_png_path}")
report_lines.append(f"  Samples CSV      : {samples_csv_path}")
report_lines.append(f"  Stats CSV        : {stats_csv_path}")
report_lines.append(f"  Priority stats   : {priority_stats_csv_path}")
report_lines.append(f"  Final stats CSV  : {final_stats_csv_path}")
report_lines.append(f"  Summary JSON     : {summary_json_path}")
report_lines.append(f"  This report      : {logs_dir / 'pipeline_report.md'}")
report_lines.append("")

# Render / sample errors
if render_errors:
    report_lines.append("## Render / sample errors")
    for method_name, error_text in render_errors.items():
        report_lines.append(f"  - {method_name}: {error_text}")
    report_lines.append("")

report_text = "\n".join(report_lines)
report_path = logs_dir / "pipeline_report.md"
report_path.write_text(report_text)

print(report_text)
print(f"\nReport saved to: {report_path}")

# Figure 5 CDF Pipeline – Run Report

Generated: 2026-04-28 08:49:37
Scenario : lab, aps3
Artifacts: /home/hieule/research/reflector-position/tmp_comparison_results2/hrbb/per_trial_runs/aps_03_seed_0044/artifacts
Seed     : 44

## Methods processed (6)
  - kmeans (kmeans)  [snapshot: best_or_fallback]
  - GA+GD (memetic)  [snapshot: final_iteration]
  - PSO+GD (pso_gd)  [snapshot: final_iteration]
  - random (random)  [snapshot: best_or_fallback]
  - random+GD (random_gd)  [snapshot: best_or_fallback]
  - weighted k-means (weighted_kmeans)  [snapshot: best_or_fallback]

## RSSI statistics – all regions
Method                    n_samples       mean     median        min         p5        p95
------------------------------------------------------------------------------------------
kmeans                         1681     -55.19     -55.74     -86.66     -70.09     -38.45
GA+GD                          1681     -54.93     -55.72     -80.96     -69.86     -38.32
PSO+GD                    